# Compare three agent frameworks

This notebook compares three agentic frameworks: smolagents, langchain and LlamaIndex. 

You'll need to install Ollama to run the LLM as you have done in the previous notebook.  

----
Code gently borrowed from Geir Freysson 
https://geirfreysson.com/posts/2025-11-09-the-ai-agent-version-of-hello-world-in-different-frameworks/

## 0. Load models with ollama

Make sure you have downloaded and installed Ollama!

In [ ]:
#Check if Ollama is installed.
import shutil

if shutil.which("ollama"):
    print("✅ Ollama is installed:", shutil.which("ollama"))
else:
    print("❌ Ollama is not installed. Download it from https://ollama.com")

In [ ]:
#show all models on your device with their sizes in GB. Requires Ollama running locally.
import ollama

result = ollama.list()
for model in result.models:
    size_gb = model.size / 1e9
    print(f"{model.model:50s} {size_gb:.2f} GB")

In [ ]:
# Download a model from Ollama if you haven't already:
!ollama pull gemma4:e2b

In [ ]:
#set the model to use for the rest of the notebook. Not every model will work with agents!
model_name = "gemma4:e2b"
#model_name = "ministral-3:3b" # a smaller model that is also faster, but less capable. 

## 1. Try smolagents

In [ ]:
!pip install smolagents 

In [ ]:
from smolagents import ToolCallingAgent, LiteLLMModel, tool

@tool
def sum_numbers(numbers: list[float]) -> float:
    """
    Calculate the sum of a list of numbers.

    Args:
        numbers: A list of numbers to sum
    """
    return sum(numbers)

@tool
def divide_sum(total: float, length: int) -> float:
    """
    Divide a sum by a length to get the average.

    Args:
        total: The sum/total to divide
        length: The number to divide by (typically the count of numbers)
    """
    return total / length

# Requires Ollama running locally: `ollama run gemma3`
model = LiteLLMModel(model_id="ollama/"+model_name)
agent = ToolCallingAgent(tools=[sum_numbers, divide_sum], model=model)
agent.run("What is the average of 10, 20, 30, 40, and 50?")

## 2. Try LangChain

In [ ]:
!pip install langchain-ollama langgraph langchain_core

In [ ]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

@tool
def sum_numbers(numbers: list[float]) -> float:
    """
    Calculate the sum of a list of numbers.

    Args:
        numbers: A list of numbers to sum
    """
    return sum(numbers)

@tool
def divide_sum(total: float, length: int) -> float:
    """
    Divide a sum by a length to get the average.

    Args:
        total: The sum/total to divide
        length: The number to divide by (typically the count of numbers)
    """
    return total / length

# Requires Ollama running locally with: ollama run ministral-3:3b
model = ChatOllama(model=model_name)
agent = create_react_agent(model, [sum_numbers, divide_sum])

# Run the agent with streaming to see intermediate steps
for chunk in agent.stream(
    {"messages": [("user", "What is the average of 10, 20, 30, 40, and 50?")]},
    stream_mode="values"
):
    chunk["messages"][-1].pretty_print()

## 3. Try Llama_index

In [ ]:
!pip install llama-index-llms-ollama

In [ ]:
from llama_index.core.agent.workflow import ReActAgent, AgentStream
from llama_index.core.tools import FunctionTool
from llama_index.llms.ollama import Ollama

def sum_numbers(numbers: list[float]) -> float:
    """
    Calculate the sum of a list of numbers.

    Args:
        numbers: A list of numbers to sum
    """
    return sum(numbers)

def divide_sum(total: float, length: int) -> float:
    """
    Divide a sum by a length to get the average.

    Args:
        total: The sum/total to divide
        length: The number to divide by (typically the count of numbers)
    """
    return total / length

sum_tool = FunctionTool.from_defaults(fn=sum_numbers)
divide_tool = FunctionTool.from_defaults(fn=divide_sum)

# Requires Ollama running locally with: ollama run ministral-3:3b
llm = Ollama(model=model_name, request_timeout=60.0)
agent = ReActAgent(tools=[sum_tool, divide_tool], llm=llm, verbose=True)

handler = agent.run("What is the average of 10, 20, 30, 40, and 50?")
async for ev in handler.stream_events():
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)
response = await handler
print(f"\n\nFinal response: {response}")